# Sketch2Shape: key-free GPU reproduction (Colab T4)
Runtime → Change runtime type → **T4 GPU**. Runs SD1.5 + ControlNet scribble → rembg → **TripoSR**, with no API keys. Takes about 8 min the first time (installs and weight downloads), then about 20–40 s per sketch.

In [ ]:
!git clone https://github.com/pranayharshithsalla/sketch2shape.git
%cd sketch2shape
!pip -q install -r requirements.txt
!pip -q install diffusers==0.35.2 transformers accelerate omegaconf einops
!pip -q install git+https://github.com/tatsy/torchmcubes.git
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
import sys; sys.path.append('/content/TripoSR')

In [ ]:
from src import gpu_local
gpu_local.register()   # adds sd_controlnet + triposr in front of the router
from src.pipeline import run
r = run('examples/mug.png', prompt='a red ceramic mug', n_candidates=3)
print(r.image_provider, r.mesh_provider, r.scores, r.timings, r.report, sep='
')

In [ ]:
from IPython.display import Image as I, display
from src.render import turntable
for p, cap in r.candidates: print(cap); display(I(p, width=200))
display(I(turntable(r.glb, r.run_dir/'turntable.png')))

In [ ]:
# Full UI with a public link (draw + upload + 3D viewer)
import app
app.build().queue().launch(share=True, debug=True)

In [ ]:
# Download the outputs
!zip -qr outputs.zip outputs
from google.colab import files; files.download('outputs.zip')